# EEG — BIOT IIIC full fine-tune (GPU)

1. **Runtime → Change runtime type → GPU**.
2. On your PC, zip the repo and upload `medical-platform.zip` to your Google Drive (`My Drive`):
   `Compress-Archive -Path e:\MASTER\medical-platform\* -DestinationPath e:\MASTER\medical-platform.zip -Force`
3. Run the cells top to bottom. Paste the output of the last eval cell back to me.

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/medical-platform && mkdir -p /content/medical-platform
!unzip -q '/content/drive/My Drive/medical-platform.zip' -d /content/medical-platform
import os
root = '/content/medical-platform'
if not os.path.exists(root + '/tools'): root = root + '/medical-platform'
print('repo root:', root)
%cd $root

In [ ]:
!pip -q install linear-attention-transformer pyarrow

Accept the HMS rules once: https://www.kaggle.com/competitions/hms-harmful-brain-activity-classification/rules — then paste your Kaggle API token (the `KGAT_...` key from Kaggle → Settings → Create New Token) into the cell below. No `kaggle.json` file needed.

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_TOKEN_HERE'   # the KGAT_... key you copied from Kaggle
print('kaggle token set' if os.environ['KAGGLE_API_TOKEN'] != 'PASTE_YOUR_TOKEN_HERE' else 'EDIT THIS CELL: paste your token first')

Download a balanced HMS subset and **full fine-tune** (encoder unfrozen). ~10–20 min download + training. If you hit out-of-memory, lower `--batch-size` to 16 or `--limit` to 10000.

In [ ]:
!python tools/train_eeg_head.py --download 2500 --hms-dir data/hms --limit 16000 --unfreeze --epochs 10 --batch-size 32 --encoder-lr 1e-5 --lr 1e-3 --seed 0 --out backend/models_weights/biot/biot_iiic.pt

In [ ]:
!python tools/eval_eeg.py --hms-dir data/hms --weights backend/models_weights/biot/biot_iiic.pt --limit 16000 --seed 0

In [ ]:
!cp backend/models_weights/biot/biot_iiic.pt '/content/drive/My Drive/biot_iiic.pt'
print('saved biot_iiic.pt to Drive')